# arXiv GraphRAG — interactive tour

This notebook shows what you can do **after** loading the arXiv knowledge graph +
vector index into AgensGraph with `prepare.py`.

A single AgensGraph database holds **both**:

- a **knowledge graph** — `(:Paper)-[:AUTHORED_BY]->(:Author)`,
  `(:Paper)-[:IN_CATEGORY]->(:Category)`, `(:Paper)-[:UPDATED_IN]->(:Year)`
- a **pgvector HNSW** index over the same `Paper` nodes (title + abstract embeddings)

**Prerequisite — load the data first** (defaults to 50k papers; lower it for a quick taste):

```bash
cd langchain
ARXIV_LIMIT=5000 .venv/bin/python examples/demos/01_arxiv_graphrag/prepare.py
```

In [1]:
import sys, pathlib

# Make the shared `_common` package importable (walk up to the demos root).
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "_common").is_dir():
    _p = _p.parent
sys.path.insert(0, str(_p))

from _common import agens, config
from _common.models import get_embeddings, get_llm
from langchain_agensgraph import AgensgraphVector

GRAPH = "arxiv"

# refresh_schema=False skips the schema-introspection scan (we only run explicit
# Cypher + vector search here), so opening even a large graph is instant.
graph = agens.make_graph(GRAPH, create=False, refresh_schema=False)
print("connected to", config.url().split("@")[-1])

connected to localhost:55432/agensgraph_demos


## The loaded graph

How big is it, and what's in it?

In [2]:
import pandas as pd

nodes = graph.query("MATCH (n) RETURN count(n) AS c")[0]["c"]
edges = graph.query("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
print(f"{nodes:,} nodes · {edges:,} edges")

pd.DataFrame(graph.query(
    "MATCH (n) RETURN label(n) AS label, count(*) AS count ORDER BY count DESC"
))

138,619 nodes · 274,748 edges


,label,count
0,Author,88455
1,Paper,50000
2,Category,147
3,Year,17


## (a) Graph analytics — pure Cypher

The graph structure answers aggregate / multi-hop questions a plain vector store can't.

In [3]:
# Most prolific authors
pd.DataFrame(graph.query('''
    MATCH (a:"Author")<-[:"AUTHORED_BY"]-(p:"Paper")
    RETURN a.name AS author, count(p) AS papers
    ORDER BY papers DESC LIMIT 10
'''))

,author,papers
0,Chablat Damien IRCCyN,84
1,Aubert B.,66
2,The BABAR Collaboration,61
3,Wenger Philippe IRCCyN,60
4,Poor H. Vincent,59
5,CDF Collaboration,41
6,Sarma S. Das,32
7,D0 Collaboration,32
8,Aaltonen T.,31
9,Gehrels N.,30


In [4]:
# Largest categories
pd.DataFrame(graph.query('''
    MATCH (c:"Category")<-[:"IN_CATEGORY"]-(p:"Paper")
    RETURN c.name AS category, count(p) AS papers
    ORDER BY papers DESC LIMIT 10
'''))

,category,papers
0,astro-ph,10250
1,hep-ph,4806
2,hep-th,4552
3,quant-ph,3155
4,gr-qc,2714
5,cond-mat.mtrl-sci,2214
6,cond-mat.stat-mech,2188
7,math-ph,2167
8,math.MP,2167
9,cond-mat.str-el,2013


In [5]:
# Top co-authorship pairs (multi-hop: author -> paper -> author).
# This is the heaviest query at full scale.
pd.DataFrame(graph.query('''
    MATCH (a1:"Author")<-[:"AUTHORED_BY"]-(p:"Paper")-[:"AUTHORED_BY"]->(a2:"Author")
    WHERE a1.name < a2.name
    RETURN a1.name AS author_1, a2.name AS author_2, count(p) AS together
    ORDER BY together DESC LIMIT 10
'''))

,author_1,author_2,together
0,Chablat Damien IRCCyN,Wenger Philippe IRCCyN,60
1,Aubert B.,The BABAR Collaboration,58
2,Aaltonen T.,CDF Collaboration,25
3,Abazov V.,D0 Collaboration,22
4,Bachmann Michael,Janke Wolfhard,19
5,Heinemeyer S.,Weiglein G.,16
6,Campana S.,Chincarini G.,16
7,Pfeiffer L. N.,West K. W.,16
8,Kartashov Yaroslav V.,Torner Lluis,15
9,Campana S.,Covino S.,15


In [6]:
# Papers per year
pd.DataFrame(graph.query('''
    MATCH (p:"Paper")-[:"UPDATED_IN"]->(y:"Year")
    RETURN y.year AS year, count(p) AS papers ORDER BY year
'''))

,year,papers
0,2007,12423
1,2008,13827
2,2009,15079
3,2010,1942
4,2011,1744
5,2012,694
6,2013,448
7,2014,749
8,2015,1290
9,2016,579


## (b) Vector semantic search — HNSW over abstracts

Reconstruct the vector store over the same `Paper` nodes (no re-embedding — every
`Paper` already has an `embedding`), then search by meaning.

In [7]:
store = AgensgraphVector.from_existing_graph(
    embedding=get_embeddings(),
    node_label="Paper",
    embedding_node_property="embedding",
    text_node_properties=["title", "abstract"],
    index_name="paper_vec",
    graph_name=GRAPH,
    engine=agens.get_engine(),
)

In [8]:
query = "quantum entanglement and information theory"
rows = []
for doc, score in store.similarity_search_with_score(query, k=5):
    title = doc.page_content.split("abstract:")[0].replace("title:", "").strip()
    rows.append({"distance": round(score, 3), "title": title[:90],
                 "arxiv_id": doc.metadata.get("id"), "year": doc.metadata.get("year")})
pd.DataFrame(rows)

,distance,title,arxiv_id,year
0,0.646,Types of quantum information,0707.3752,2009
1,0.642,A paradigm for entanglement theory based on qu...,0801.0458,2008
2,0.635,Quantum Teleportation and Von Neumann Entropy,0707.1227,2007
3,0.629,Generalized information theoretic measure to d...,0707.2195,2008
4,0.625,Smooth Renyi Entropies and the Quantum Informa...,0801.0282,2009


## (c) Hybrid GraphRAG — retrieve, then expand through the graph

Vector search finds semantically relevant papers; the **graph** then expands to
papers sharing an author. Both feed a grounded LLM answer with citations.

In [9]:
from psycopg.types.json import Jsonb

def graphrag(question, k=5):
    seeds = store.similarity_search(question, k=k)
    seed_ids = [d.metadata["id"] for d in seeds]
    related = graph.query(
        'UNWIND %(ids)s AS pid '
        'MATCH (p:"Paper" {id: pid})-[:"AUTHORED_BY"]->(:"Author")<-[:"AUTHORED_BY"]-(rel:"Paper") '
        'WHERE rel.id <> pid RETURN DISTINCT rel.title AS title LIMIT 8',
        {"ids": Jsonb(seed_ids)},
    )
    context = "\n\n".join(f"- {d.page_content}" for d in seeds)
    if related:
        context += "\n\nRelated work (same authors):\n" + "\n".join(f"- {r['title']}" for r in related)
    prompt = (
        "Using ONLY the arXiv abstracts below, answer concisely and cite paper "
        f"titles you rely on.\n\nQuestion: {question}\n\nAbstracts:\n{context}\n\nAnswer:"
    )
    print(f"vector seeds: {', '.join(seed_ids)}   |   graph-expanded papers: {len(related)}\n")
    print(get_llm().invoke(prompt).content)

graphrag("What methods are used for studying black hole thermodynamics?")

vector seeds: 0704.3102, 0801.3583, 0709.1812, 0708.3145, 0711.2330   |   graph-expanded papers: 8



The methods used for studying black hole thermodynamics include:

1. **Geometrothermodynamics**: This method reformulates black hole thermodynamics using a geometric approach that is invariant under Legendre transformations, allowing for the exploration of phase transitions and critical points in black hole thermodynamic systems (see "Geometrothermodynamics of black holes").

2. **Tunneling Interpretation of Hawking Radiation**: This approach derives the relationship between black hole temperature and surface gravity, applicable even in the presence of quantum effects, and includes graphical analyses of temperature variations (see "Noncommutative Black Hole Thermodynamics").

3. **Semiclassical Models**: A method that employs analogies to Bohr's quantization to evaluate thermodynamic attributes such as entropy and temperature, providing estimates for black hole evaporation times (see "A Bohr's Semiclassical Model of the Black Hole Thermodynamics").

4. **Nonequilibrium Thermodynamics**

## What you can do with this

After one load you have a single AgensGraph database that supports:

- **Structural / analytical queries** (Cypher) — collaboration networks, trends by
  year, category co-occurrence, paths between authors, …
- **Semantic search** (pgvector HNSW) — find papers by meaning, with scores and
  metadata.
- **GraphRAG** — combine the two: retrieve by similarity, expand by relationships,
  and ground an LLM answer — all without leaving the database.

Try your own:

```python
graphrag("your question here")
store.similarity_search("a topic you care about", k=8)
```

When you're finished, close the shared connection pool:

```python
agens.close()
```